# Solutions — Typing, Dataclasses, Enums, singledispatch & Type Introspection

Exercise notebook: `10_advanced_python/02_typing_dataclasses_enums.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Enums for statuses

**Working solution**

In [ ]:
from enum import Enum

class OrderStatus(str, Enum):
    PLACED = "PLACED"
    SHIPPED = "SHIPPED"
    DELIVERED = "DELIVERED"
    CANCELLED = "CANCELLED"
    RETURNED = "RETURNED"

    @classmethod
    def parse(cls, value):
        try:
            return cls(str(value).strip().upper())
        except ValueError:
            raise ValueError(f"invalid status {value!r}; expected one of {[m.value for m in cls]}") from None

    @property
    def is_terminal(self):
        return self in (OrderStatus.DELIVERED, OrderStatus.CANCELLED, OrderStatus.RETURNED)

**Explanation**

Enums replace magic strings with a closed set the IDE can autocomplete. The `str` mixin keeps JSON/CSV interop painless (`json.dumps(OrderStatus.PLACED)` works). Python 3.11 also offers `StrEnum`.

**Complexity:** O(1).

**Common mistakes**

- Comparing with `is` against strings.
- Scattering `if status in ('DELIVERED', 'CANCELLED', ...)` everywhere.

## Exercise 2 — 🟡 Medium · A validated, frozen, slotted value type

**Working solution**

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True, slots=True, order=True)
class OrderLine:
    order_id: str
    line_no: int
    product_id: str
    quantity: int
    unit_price: Decimal

    def __post_init__(self):
        if not isinstance(self.unit_price, Decimal):
            if isinstance(self.unit_price, float):
                raise TypeError("unit_price must not be a float")
            object.__setattr__(self, "unit_price", Decimal(str(self.unit_price)))
        if not self.order_id.strip() or not self.product_id.strip():
            raise ValueError("ids must be non-blank")
        if self.line_no < 1 or self.quantity < 1:
            raise ValueError("line_no and quantity must be >= 1")
        if self.unit_price < 0:
            raise ValueError("unit_price must be >= 0")

    @property
    def amount(self) -> Decimal:
        return self.quantity * self.unit_price

    @classmethod
    def from_dict(cls, d: dict) -> "OrderLine":
        return cls(str(d["order_id"]), int(d["line_no"]), str(d["product_id"]), int(d["quantity"]), d["unit_price"])

**Explanation**

`frozen` = immutable + hashable; `slots` = no per-instance `__dict__` (less memory, typo-proof attributes); `order` = comparison by field tuple. Coercion inside a frozen dataclass needs
`object.__setattr__` — the documented escape hatch.

**Complexity:** O(1).

**Common mistakes**

- `self.unit_price = Decimal(...)` in a frozen dataclass → `FrozenInstanceError`.

## Exercise 3 — 🟡 Medium · Function overloading with singledispatch

**Working solution**

In [ ]:
import io
from functools import singledispatch
from pathlib import Path

def _clean(lines):
    return [line.strip() for line in lines if line.strip()]

@singledispatch
def read_lines(source):
    if hasattr(source, "read"):
        return _clean(source.read().splitlines())
    return _clean(source)

@read_lines.register
def _(source: Path):
    return _clean(source.read_text(encoding="utf-8").splitlines())

@read_lines.register
def _(source: str):
    return _clean(source.splitlines())

@read_lines.register
def _(source: bytes):
    return _clean(source.decode("utf-8").splitlines())

**Explanation**

`singledispatch` picks the implementation by the **runtime type of the first argument**, using the annotation in `register`. Subclasses (e.g. `PosixPath`) dispatch to the `Path` handler.

**Complexity:** O(n).

**Common mistakes**

- A long `if isinstance(...) elif isinstance(...)` chain that every new type must edit.

## Exercise 4 — 🟠 Hard · Convert raw rows using a dataclass's type hints

**Working solution**

In [ ]:
import dataclasses
import types
import typing
from datetime import date, datetime
from decimal import Decimal
from enum import Enum

def _bool(s):
    low = s.strip().lower()
    if low in ("true", "1", "yes"):
        return True
    if low in ("false", "0", "no"):
        return False
    raise ValueError(f"not a bool: {s!r}")

_SIMPLE = {str: str, int: int, float: float, bool: _bool, Decimal: Decimal, date: date.fromisoformat,
           datetime: lambda s: datetime.fromisoformat(s.replace("Z", "+00:00"))}

def _convert(tp, raw, name):
    origin = typing.get_origin(tp)
    if origin in (typing.Union, types.UnionType):
        args = [a for a in typing.get_args(tp) if a is not type(None)]
        if raw is None or str(raw).strip() == "":
            return None
        return _convert(args[0], raw, name)
    if raw is None or str(raw).strip() == "":
        raise ValueError(f"{name} must not be blank")
    if isinstance(tp, type) and issubclass(tp, Enum):
        return tp(raw.strip())
    if tp in _SIMPLE:
        return _SIMPLE[tp](raw.strip() if isinstance(raw, str) else raw)
    raise TypeError(f"unsupported type for {name}: {tp!r}")

def from_row(cls, row):
    hints = typing.get_type_hints(cls)
    kwargs, missing = {}, []
    for f in dataclasses.fields(cls):
        if f.name not in row:
            has_default = f.default is not dataclasses.MISSING or f.default_factory is not dataclasses.MISSING
            if not has_default:
                missing.append(f.name)
            continue
        kwargs[f.name] = _convert(hints[f.name], row[f.name], f.name)
    if missing:
        raise TypeError(f"missing fields: {missing}")
    return cls(**kwargs)

**Explanation**

`get_type_hints` resolves annotations (including string ones); `get_origin`/`get_args` decompose `int | None` into `Union` + its members. This is exactly how pydantic,
FastAPI and dataclass-based config loaders turn type hints into runtime behaviour.

**Complexity:** O(fields).

**Common mistakes**

- Reading `f.type` directly — it may be a *string* when `from __future__ import annotations` is used.
- Treating `bool('false')` as False.

## Challenge — 🔴 Very Hard · Typed nested configuration (pydantic-lite)

**Working solution**

In [ ]:
import dataclasses
import types
import typing
from enum import Enum

class ConfigError(ValueError):
    pass

def _type_name(tp):
    return getattr(tp, "__name__", str(tp))

def load_typed(cls, data, path=""):
    here = path or "$"
    origin, args = typing.get_origin(cls), typing.get_args(cls)
    if origin in (typing.Union, types.UnionType):
        if data is None and type(None) in args:
            return None
        non_none = [a for a in args if a is not type(None)]
        return load_typed(non_none[0], data, path)
    if dataclasses.is_dataclass(cls):
        if not isinstance(data, dict):
            raise ConfigError(f"{here}: expected object, got {type(data).__name__}")
        fields = {f.name: f for f in dataclasses.fields(cls)}
        unknown = sorted(set(data) - set(fields))
        if unknown:
            raise ConfigError(f"{here}: unknown keys {unknown}")
        hints = typing.get_type_hints(cls)
        kwargs = {}
        for name, f in fields.items():
            sub = f"{path}.{name}" if path else name
            if name not in data:
                if f.default is dataclasses.MISSING and f.default_factory is dataclasses.MISSING:
                    raise ConfigError(f"{sub}: missing required field")
                continue
            kwargs[name] = load_typed(hints[name], data[name], sub)
        return cls(**kwargs)
    if origin is list:
        if not isinstance(data, list):
            raise ConfigError(f"{here}: expected list, got {type(data).__name__}")
        return [load_typed(args[0], item, f"{path}[{i}]") for i, item in enumerate(data)]
    if origin is dict:
        if not isinstance(data, dict):
            raise ConfigError(f"{here}: expected object, got {type(data).__name__}")
        return {k: load_typed(args[1], v, f"{path}.{k}" if path else k) for k, v in data.items()}
    if isinstance(cls, type) and issubclass(cls, Enum):
        try:
            return cls(data)
        except ValueError:
            raise ConfigError(f"{here}: expected one of {[m.value for m in cls]}, got {data!r}") from None
    if cls is float and isinstance(data, int) and not isinstance(data, bool):
        return float(data)
    if cls in (int, float, str, bool):
        if isinstance(data, bool) and cls is not bool:
            raise ConfigError(f"{here}: expected {_type_name(cls)}, got bool")
        if not isinstance(data, cls):
            raise ConfigError(f"{here}: expected {_type_name(cls)}, got {type(data).__name__}")
        return data
    raise ConfigError(f"{here}: unsupported type {cls!r}")

**Explanation**

One recursive function dispatching on the *shape of the type* (dataclass / list / dict / Optional / Enum / scalar) and threading a path string for error messages. The result is
a fully typed, IDE-friendly config object, validated at startup instead of failing at 2 a.m. halfway through a job.

**Complexity:** O(size of config).

**Common mistakes**

- `isinstance(True, int)` is True — reject bools explicitly.
- Error messages without a path: 'expected int' — where?